# Explore GSE176031 — Pancreatic Cancer T cells (PDAC)

**Dataset**: Pancreatic adenocarcinoma (PDAC) scRNA-seq T cell co-culture study  
Raw DGE matrices from 53 samples covering:

| Sample type | Files | Notes |
|---|---|---|
| Fresh TILs — tumor | `PR5xxx_T1`, `PR5xxx_T2` | Tumor-infiltrating T cells |
| Fresh TILs — normal | `PR5xxx_N1`, `PR5xxx_N2` | Adjacent normal tissue |
| Peripheral blood | `PB1A`, `PB1B`, `PB2A`, `PB2B`, `AUG_PB` | Circulating T cells |
| Tumor organoid co-culture | `PR5xxx_T_org` | T cells co-cultured with tumor PDOs |
| Normal organoid co-culture | `PR5xxx_N_org` | T cells co-cultured with normal PDOs |
| Pooled stimulation | `PR5186`, `PR5196`, `PR5199`, `PR5269` | In-vitro expanded/stimulated |

**Key question**: Which T cells are tumor-reactive?  
**Strategy**: Cells in `T_org` samples that are activated (CD69+, IFNG+, Ki67+) but  
NOT in matched `N_org` → these responded to tumor-specific antigens in organoids.

**Protocol note**: Drop-seq / inDrop short 12-nt barcodes, ~1000–1300 cells per file,  
~22K genes — enriched T cell fraction (not whole tumour).


In [ ]:
suppressPackageStartupMessages({
  .libPaths(c("renv/library/macos/R-4.4/aarch64-apple-darwin20", .libPaths()))
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(patchwork)
})

options(repr.plot.width = 12, repr.plot.height = 5)

# ── Extract all DGE files from the tar ────────────────────────────────────────
TAR    <- "data/raw_downloads/GSE176031_RAW.tar"
OUTDIR <- "data/gse176031_dge"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

if (length(list.files(OUTDIR, pattern = "\\.gz$")) < 53) {
  cat("Extracting tar ...\n")
  system(paste("tar -xf", TAR, "-C", OUTDIR))
  cat("Done.\n")
} else {
  cat("Already extracted —", length(list.files(OUTDIR, pattern="\\.gz$")), "files\n")
}
list.files(OUTDIR, pattern = "\\.gz$") |> head(6)


## 1. Parse Sample Metadata from Filenames

In [ ]:
files <- list.files(OUTDIR, pattern = "_dge\\.txt\\.gz$", full.names = TRUE)
cat(length(files), "DGE files\n\n")

# Parse metadata from filename
parse_meta <- function(f) {
  bn <- basename(f)
  bn <- sub("GSM[0-9]+_", "", bn)      # strip GSM ID
  bn <- sub("_dge\\.txt\\.gz$", "", bn) # strip suffix
  bn <- sub("^PA_", "", bn)             # strip PA_ prefix
  bn <- sub("^HNW_", "", bn)            # strip HNW_ (batch marker)
  
  # Classify sample type
  sample_type <- dplyr::case_when(
    grepl("_T_org|_T_V[0-9]_org|_T_base_org", bn) ~ "tumor_organoid",
    grepl("_N_org|_N_V[0-9]_org|_N_base_org", bn) ~ "normal_organoid",
    grepl("^PB|^AUG_PB", bn)                       ~ "peripheral_blood",
    grepl("_T[0-9]_", bn)                          ~ "tumor_tissue",
    grepl("_N[0-9]_", bn)                          ~ "normal_tissue",
    TRUE                                            ~ "pooled_stimulation"
  )
  
  # Patient ID
  patient <- dplyr::case_when(
    grepl("PR5249", bn) ~ "PR5249",
    grepl("PR5251", bn) ~ "PR5251",
    grepl("PR5254", bn) ~ "PR5254",
    grepl("PR5261", bn) ~ "PR5261",
    grepl("PR5269", bn) ~ "PR5269",
    grepl("PR5274", bn) ~ "PR5274",
    grepl("PR5316", bn) ~ "PR5316",
    grepl("PR5186", bn) ~ "PR5186",
    grepl("PR5196", bn) ~ "PR5196",
    grepl("PR5199", bn) ~ "PR5199",
    grepl("^PB1",   bn) ~ "PB_patient1",
    grepl("^PB2",   bn) ~ "PB_patient2",
    grepl("^AUG",   bn) ~ "PB_AUG",
    TRUE                ~ "unknown"
  )
  
  data.frame(file=f, sample_id=bn, patient=patient, sample_type=sample_type,
             stringsAsFactors=FALSE)
}

meta_df <- do.call(rbind, lapply(files, parse_meta))
cat("=== Sample composition ===\n")
print(table(meta_df$sample_type))
cat("\n=== Patients with organoid data ===\n")
print(table(meta_df$patient[grepl("organoid", meta_df$sample_type)]))


## 2. Load All DGE Matrices → Seurat Objects

In [ ]:
# Load one Seurat object per file, add metadata, merge
load_dge <- function(row) {
  mat <- read.table(row$file, header=TRUE, row.names=1, check.names=FALSE)
  # Transpose: DGE is genes×cells, Seurat needs cells×genes as input but
  # Read10X-style dgCMatrix is genes×cells — keep as-is
  mat <- as.sparse(as.matrix(mat))
  obj <- CreateSeuratObject(counts = mat, min.features = 50,
                            project = row$sample_id)
  obj$patient     <- row$patient
  obj$sample_type <- row$sample_type
  obj$sample_id   <- row$sample_id
  obj$orig.ident  <- row$sample_id
  obj
}

cat("Loading", nrow(meta_df), "DGE files ...\n")
seurat_list <- lapply(seq_len(nrow(meta_df)), function(i) {
  if (i %% 10 == 0) cat("  loaded", i, "/", nrow(meta_df), "\n")
  load_dge(meta_df[i,])
})
names(seurat_list) <- meta_df$sample_id

# Summary
ncells <- sapply(seurat_list, ncol)
cat("\nCells per sample (range):", min(ncells), "–", max(ncells), "\n")
cat("Total cells:", sum(ncells), "\n")


## 3. QC Filtering

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

# Compute mitochondrial % for each object
seurat_list <- lapply(seurat_list, function(obj) {
  obj[["pct.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
  obj
})

# Merge for QC visualisation
all_meta <- do.call(rbind, lapply(names(seurat_list), function(nm) {
  m <- seurat_list[[nm]]@meta.data
  m$sample_id <- nm; m
}))

cat("QC before filtering:\n")
cat("  nFeature_RNA median:", median(all_meta$nFeature_RNA), "\n")
cat("  pct.mt median:", round(median(all_meta$pct.mt),1), "%\n")

p1 <- ggplot(all_meta, aes(x=sample_type, y=nFeature_RNA, fill=sample_type)) +
  geom_boxplot(show.legend=FALSE, outlier.size=0.3) +
  coord_flip() + theme_classic(base_size=10) +
  labs(title="Genes detected", x=NULL)

p2 <- ggplot(all_meta, aes(x=sample_type, y=pct.mt, fill=sample_type)) +
  geom_boxplot(show.legend=FALSE, outlier.size=0.3) +
  coord_flip() + theme_classic(base_size=10) +
  labs(title="% mitochondrial", x=NULL)

p1 + p2


In [ ]:
# Apply QC filters
seurat_list_filt <- lapply(seurat_list, function(obj) {
  subset(obj, subset = nFeature_RNA > 200 & nFeature_RNA < 6000 & pct.mt < 25)
})
ncells_before <- sum(sapply(seurat_list, ncol))
ncells_after  <- sum(sapply(seurat_list_filt, ncol))
cat("Cells before filtering:", ncells_before, "\n")
cat("Cells after filtering: ", ncells_after, "(", round(100*ncells_after/ncells_before), "% retained)\n")


## 4. Normalise, Merge, Cluster

In [ ]:
# Normalise each object
seurat_list_filt <- lapply(seurat_list_filt, function(obj) {
  NormalizeData(obj, verbose=FALSE) |>
    FindVariableFeatures(nfeatures=2000, verbose=FALSE) |>
    ScaleData(verbose=FALSE)
})

# Merge into one object
cat("Merging ...\n")
merged <- merge(seurat_list_filt[[1]], y = seurat_list_filt[-1],
                add.cell.ids = names(seurat_list_filt),
                project = "GSE176031")

cat("Total cells in merged object:", ncol(merged), "\n")
cat("Sample types:\n")
print(table(merged$sample_type))


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

# Re-normalise merged object and reduce
merged <- NormalizeData(merged, verbose=FALSE)
merged <- FindVariableFeatures(merged, nfeatures=3000, verbose=FALSE)
merged <- ScaleData(merged, verbose=FALSE)
merged <- RunPCA(merged, npcs=30, verbose=FALSE)

# Cluster + UMAP
merged <- FindNeighbors(merged, dims=1:20, verbose=FALSE)
merged <- FindClusters(merged,  resolution=0.5, verbose=FALSE)
merged <- RunUMAP(merged, dims=1:20, verbose=FALSE)

p_type <- DimPlot(merged, group.by="sample_type", pt.size=0.3) +
  labs(title="UMAP — Sample type") + theme_void(base_size=10)
p_clus <- DimPlot(merged, label=TRUE, pt.size=0.3) +
  labs(title="UMAP — Clusters") + theme_void(base_size=10)
p_type | p_clus


## 5. Cell-type Annotation — T-cell Markers

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 10)

# Key markers: T cell, NK, Myeloid, Epithelial
markers <- c(
  # T cells
  "CD3D","CD3E","CD8A","CD4",
  # Activation / exhaustion
  "CD69","HAVCR2","LAG3","PDCD1","CTLA4",
  # Effector
  "IFNG","GZMB","PRF1","TNF",
  # Proliferation
  "MKI67","TOP2A",
  # NK
  "NCAM1","KLRD1",
  # Myeloid
  "CD14","LYZ","CST3"
)
FeaturePlot(merged, features=markers, ncol=6, pt.size=0.1,
            order=TRUE, cols=c("lightgrey","#e63946"))


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

# Violin plots per cluster for key T cell markers
VlnPlot(merged, features=c("CD3D","CD8A","CD4","HAVCR2","MKI67","IFNG"),
        ncol=6, pt.size=0, fill.by="ident")


## 6. Tumor-reactive T cells — Organoid Co-culture Comparison

In [ ]:
cat("=== Cells in organoid co-culture experiments ===\n")
print(table(merged$sample_type[grepl("organoid", merged$sample_type)]))

# Subset to organoid samples only
org_cells <- subset(merged, subset = sample_type %in% c("tumor_organoid","normal_organoid"))
cat("\nOrganoid co-culture cells:", ncol(org_cells), "\n")
cat("  Tumor organoid:", sum(org_cells$sample_type == "tumor_organoid"), "\n")
cat("  Normal organoid:", sum(org_cells$sample_type == "normal_organoid"), "\n")


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

# Compare activation/exhaustion markers between tumor vs normal organoids
activation_genes <- c("CD69","MKI67","IFNG","GZMB","PRF1","TNF",
                       "HAVCR2","LAG3","PDCD1","ENTPD1")

# Filter to genes present
activation_genes <- activation_genes[activation_genes %in% rownames(org_cells)]

VlnPlot(org_cells, features = head(activation_genes, 8),
        group.by = "sample_type", ncol = 4, pt.size = 0,
        cols = c("normal_organoid"="#457b9d","tumor_organoid"="#e63946")) +
  plot_annotation(title = "Activation markers: Tumor vs Normal organoid T cells")


In [ ]:
# Differential expression: tumor_organoid vs normal_organoid
Idents(org_cells) <- "sample_type"
de_org <- FindMarkers(org_cells,
                      ident.1 = "tumor_organoid",
                      ident.2 = "normal_organoid",
                      logfc.threshold = 0.25,
                      test.use = "wilcox",
                      verbose = FALSE)
de_org <- de_org[order(de_org$avg_log2FC, decreasing=TRUE),]

cat("=== TOP 20 genes upregulated in TUMOR organoid T cells ===\n")
print(head(de_org[de_org$p_val_adj < 0.05,], 20))


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

# Volcano plot
de_org$gene   <- rownames(de_org)
de_org$signif <- de_org$p_val_adj < 0.05
de_org$label  <- ifelse(abs(de_org$avg_log2FC) > 1 & de_org$signif,
                         de_org$gene, "")

ggplot(de_org, aes(avg_log2FC, -log10(p_val_adj + 1e-300),
                   colour = signif)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_colour_manual(values = c("grey70","#e63946")) +
  ggrepel::geom_text_repel(aes(label = label), size = 2.5,
                            max.overlaps = 20, segment.colour="grey60") +
  geom_vline(xintercept = c(-1,1), lty=2, colour="grey50") +
  theme_classic(base_size=11) +
  labs(title="DE: Tumor organoid vs Normal organoid T cells",
       x="log2 FC (tumor / normal)", y="-log10 adj. p-value",
       colour="FDR<0.05")


## 7. Fresh TIL Comparison — Tumor vs Normal Tissue

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

til_cells <- subset(merged, subset = sample_type %in% c("tumor_tissue","normal_tissue"))
cat("Fresh TIL cells:", ncol(til_cells), "\n")

Idents(til_cells) <- "sample_type"
VlnPlot(til_cells,
        features = c("CD69","HAVCR2","LAG3","PDCD1","GZMB","MKI67","IFNG","TNF"),
        group.by = "sample_type", ncol = 4, pt.size = 0,
        cols = c("normal_tissue"="#457b9d","tumor_tissue"="#e63946")) +
  plot_annotation(title = "Exhaustion / activation: Tumor vs Normal TILs")


## 8. Per-patient Clonal Landscape

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)

# Cells per patient × sample_type
pat_tab <- as.data.frame(table(merged$patient, merged$sample_type))
colnames(pat_tab) <- c("patient","sample_type","n")
pat_tab <- pat_tab[pat_tab$n > 0,]
pat_tab <- pat_tab[!grepl("pooled", pat_tab$sample_type),]

p_bar <- ggplot(pat_tab, aes(x=patient, y=n, fill=sample_type)) +
  geom_col(position="dodge", alpha=0.85) +
  scale_fill_brewer(palette="Set2") +
  theme_classic(base_size=10) +
  theme(axis.text.x=element_text(angle=35, hjust=1)) +
  labs(title="Cells per patient × sample type", x=NULL, y="Cells", fill=NULL)
print(p_bar)


## 9. Summary

In [ ]:
cat("╔══════════════════════════════════════════════════════════════════╗\n")
cat("║  GSE176031 PDAC T-cell dataset — exploration summary            ║\n")
cat("╚══════════════════════════════════════════════════════════════════╝\n\n")

cat("Total cells (post-QC) :", ncol(merged), "\n")
cat("Clusters              :", length(levels(merged$seurat_clusters)), "\n")
cat("Sample types          :\n")
for (nm in names(sort(table(merged$sample_type), decreasing=TRUE))) {
  cat("  ", nm, ":", table(merged$sample_type)[nm], "\n")
}
cat("\nTumor-reactive T cell proxy:\n")
cat("  → 'tumor_organoid' cells with high CD69/IFNG/GZMB/MKI67\n")
cat("    and low expression in matched 'normal_organoid' condition\n")
cat("  → DE analysis identifies transcriptionally activated cells\n")
cat("\nNote: No VDJ/TCR sequencing data in this GEO submission.\n")
cat("  → If TCR data exists, request from corresponding author\n")
cat("  → Alternatively use TRUST4/TraCeR to reconstruct from mRNA reads\n")
